In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

from calc_cavity import *
import glob

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/PHI_deadpix_maskcorrected.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
folders = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/fdt/calibration/*'))
folders

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-03-30',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-10-16',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-10-27',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-12-02',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-01-19',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-15',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25']

In [4]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/'

In [5]:
cavity = calc_cavity(folder, dark_file=dark_file,
                             prefilter_file=prefilter_file,
                             deadpix_file=deadpix_file,
                             distortion_file=distortion_file)

looking for files in folder: /home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/
found 9 input files
first input file is: /home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T040003_V202606241235C_0663100100.fits.gz
last input file is: /home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T043205_V202606241634C_0663100300.fits.gz
reading and preprocessing the data
dark signal file is: /home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz
dead pixels file is: /home/ulyanov/data/solo/phi/dead_pixels/PHI_deadpix_maskcorrected.fits
prefilter file is: /home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt
distortion file is: /home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz
distance is: 0.32765725 AU
PMP SP temperature is: 45 C
FG SP temperature is: 66 C
continuum position is: blue
wavelengths are: [61

/home/ulyanov/PycharmProjects/flatfield/src/utils.py:312: RuntimeWarning: invalid value encountered in divide
  with np.errstate(invalid='ignore'):


done


In [6]:
plt.figure(figsize=(10,10))
plt.imshow(cavity, 'bwr', vmin=-0.05, vmax=0.05)
plt.tight_layout()

In [5]:
files = sorted(glob.glob(folder + '/*.fits.gz'))

datas = []
for file in files:
    with fits.open(file) as hdul:
        header = hdul[0].header
        data = hdul[0].data

    data = preprocess(data, header, dark_file=dark_file,
                             prefilter_file=prefilter_file,
                             deadpix_file=deadpix_file,
                             distortion_file=distortion_file)
    datas.append(data)

datas = np.array(datas)

In [22]:
shifts = np.array([get_wv_shift(data, cpos=5, delta_wv=0.069, log=True) for data in datas])
images = datas[:, -1]

centers = []
for image in images:
    xc, yc, rsun = find_center(image)
    centers.append([xc, yc])
centers = np.array(centers)

/tmp/ipykernel_105235/765496226.py:5: RuntimeWarning: divide by zero encountered in log
  temp = -np.log(temp[cpos] - np.delete(temp, cpos, axis=0))
/tmp/ipykernel_105235/765496226.py:5: RuntimeWarning: invalid value encountered in log
  temp = -np.log(temp[cpos] - np.delete(temp, cpos, axis=0))
/tmp/ipykernel_105235/765496226.py:11: RuntimeWarning: invalid value encountered in subtract
  b, c = (r - l) / 2, (l + r) - 2 * a


In [55]:
def kll(data, shifts, weights, niter=20, sigma=1,
        verbose=False, **kwargs):

    F = np.zeros_like(data[0])
    C0 = shifts[0]
    for iter in range(niter):

        # calculating averaged image in the position of first image
        D = np.zeros_like(F)
        W = np.zeros_like(F)
        for Di, Ci, Wi in zip(data, shifts, weights):
            Di_ = roll_float(Di - F, *(C0 - Ci), **kwargs)
            Wi_ = roll_float(Wi, *(C0 - Ci), **kwargs)

            W += Wi_
            with np.errstate(invalid='ignore'):
                D += np.nan_to_num((Di_ - D) * Wi_ / W)

        F_ = np.zeros_like(F)
        W = np.zeros_like(F)

        for Di, Ci, Wi in zip(data, shifts, weights):
            Di_ = Di - roll_float(D, *(Ci - C0), **kwargs)

            delta = np.sqrt(np.mean(Wi * (Di_ - F) ** 2) / np.mean(Wi))
            Wi_ = Wi / np.abs(Di_ - F).clip(delta * sigma)

            W += Wi_
            with np.errstate(invalid='ignore'):
                F_ += np.nan_to_num((Di_ - F_) * Wi_ / W)

        F = np.nan_to_num(F_)

    if verbose:
        print('')

    return F

In [58]:
cavity = kll(np.nan_to_num(shifts), centers, images.clip(0), niter=10, sigma=1)

In [61]:
plt.figure(figsize=(10,10))
plt.imshow(cavity, 'bwr', vmin=-0.05, vmax=0.05)
plt.tight_layout()

In [60]:
plt.figure(figsize=(10,10))
plt.imshow(shifts[0] - cavity, 'bwr', vmin=-0.05, vmax=0.05)
plt.tight_layout()